In [ ]:
import copy
import math
import torch
import numpy as np
import torch.nn as nn
from torch import Tensor
from functools import partial
import torch.nn.functional as F
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple, Union
from torchvision.models.resnet import BasicBlock, Bottleneck, conv1x1
import ee
import pandas as pd
import openpyxl
import ee
import datetime
import os
from PIL import Image
import numpy as np
import json
import IPython.display as disp
import os
import json
import torch
import numpy as np
from PIL import Image
from skimage.io import imread
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import transforms
from skimage.io import imread
import matplotlib.pyplot as plt

In [ ]:
from torchvision.models.resnet import (
    ResNet18_Weights,
    ResNet34_Weights,
    ResNet50_Weights,
    ResNet101_Weights,
)
import torch
from torch import nn
from torch.nn import functional as F

In [ ]:
class ImagePadder:
    def __init__(
        self,
        dir_images,
        pad_left=15,
        pad_right=15,
        pad_top=11,
        pad_bottom=11,
        file_anchor_image=None,  # Make this optional
        anchor_image_shape=(1250, 650, 3)  # Default shape, modify as needed
    ):
        if file_anchor_image:
            self._anchor_image = imread(os.path.join(dir_images, file_anchor_image))
        else:
            self._anchor_image = np.zeros(anchor_image_shape, dtype=np.uint8)  # Blank image
        self._anchor_image_shape = self._anchor_image.shape
        self._pad_left = pad_left
        self._pad_right = pad_right
        self._pad_top = pad_top
        self._pad_bottom = pad_bottom
        self._anchor_image_resized = None
        self._anchor_image_resized_shape = None

        self._set_anchor_image_resized()

    def _set_anchor_image_resized(self):
        height, width = self._anchor_image.shape[:2]
        target_width = self._pad_left + width + self._pad_right
        target_height = self._pad_top + height + self._pad_bottom
        self._anchor_image_resized = cv2.resize(
            self._anchor_image[:, 260:, :],
            (target_width, target_height),
            interpolation=cv2.INTER_LINEAR,
        )
        self._anchor_image_resized_shape = self._anchor_image_resized.shape

    def pad_image(self, image):
        padded_image = self._anchor_image_resized.copy()
        padded_image[
            self._pad_top : self._anchor_image_resized_shape[0] - self._pad_bottom,
            self._pad_left : self._anchor_image_resized_shape[1] - self._pad_right,
            :,
        ] = image
        return padded_image

    def pad_label(self, label):
        padded_label = np.pad(
            label,
            ((self._pad_top, self._pad_bottom), (self._pad_left, self._pad_right)),
        )
        return padded_label


In [ ]:
class M4DSAROilSpillDataset(Dataset):
    def __init__(
        self,
        dir_data,
        list_images,
        which_set="train",
        file_stats_json="image_stats.json",
    ):
        self.dir_data = dir_data
        self.which_set = which_set
        self.file_stats_json = file_stats_json

        # Load image statistics (mean, std)
        self.dict_stats = self._load_stats()

        # Define paths for images and labels
        self._dir_images = os.path.join(self.dir_data, "images")
        self._dir_labels = os.path.join(self.dir_data, "labels_1D")

        # Sort images and labels
        self._list_images = sorted(list_images)
        self._list_labels = [f.replace(".jpg", ".png") for f in self._list_images]

        # Initialize ImagePadder
        if self.which_set == "train":
            file_anchor_image = "img_0814.jpg"  # Training image
        else:
            file_anchor_image = self._list_images[0]  # First test image or a generic image

        # Initialize ImagePadder with the selected anchor image
        self._image_padder = ImagePadder(
            dir_images=self._dir_images, file_anchor_image=file_anchor_image
        )
        # Define image transformations
        self._image_transform = transforms.Compose(
            [
                transforms.ToPILImage(),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[
                        self.dict_stats["mean"],
                        self.dict_stats["mean"],
                        self.dict_stats["mean"],
                    ],
                    std=[
                        self.dict_stats["std"],
                        self.dict_stats["std"],
                        self.dict_stats["std"],
                    ],
                ),
            ]
        )

        # Define augmentations for the training set
        if self.which_set == "train":
            self._affine_transform = transforms.Compose(
                [
                    transforms.RandomHorizontalFlip(),
                    transforms.RandomVerticalFlip(),
                ]
            )

    def _load_stats(self):
        return {"mean": 0.5185, "std": 0.197}

    def __len__(self):
        return len(self._list_images)

    def __getitem__(self, idx, return_original=False):
        file_image = os.path.join(self._dir_images, self._list_images[idx])
        file_label = os.path.join(self._dir_labels, self._list_labels[idx])

        image = imread(file_image)
        label = imread(file_label)

        original_image = image.copy()

        # Pad image and label
        image = self._image_padder.pad_image(image)
        label = self._image_padder.pad_label(label)

        if self.which_set == "train":
            image_tensor = torch.from_numpy(image)
            label_tensor = torch.from_numpy(label).unsqueeze(dim=-1)

            # Combine and apply augmentation
            stacked = torch.cat([image_tensor, label_tensor], dim=-1)
            stacked = torch.permute(stacked, [2, 0, 1])
            stacked_transformed = self._affine_transform(stacked)
            stacked_transformed = torch.permute(stacked_transformed, [1, 2, 0])
            stacked_arr = stacked_transformed.numpy()

            image = stacked_arr[:, :, :-1]
            label = stacked_arr[:, :, -1]

        # Apply normalization transform
        image = self._image_transform(image)

        if return_original:
            return image, label, original_image
        else:
            return image, label


In [ ]:
# Set the paths to your dataset
dir_data = '/content/drive/MyDrive/CIMA2023/Documentos2023/Proyectos/Proy1-ImagenesEspectrales/data/imagenes/OilDataset/train/'
list_images = os.listdir(os.path.join(dir_data, "images"))

In [ ]:
train_dataset = M4DSAROilSpillDataset(dir_data, list_images, which_set="train")

In [ ]:
def get_dataloaders_for_training(
    dir_dataset, batch_size, random_state=None, num_workers=4
):
    """
    ---------
    Arguments
    ---------
    dir_dataset : str
        full path to dataset directory
    batch_size : int
        batch size to be used
    random_state : int
        random state to be used for train / validation set split (default: None)
    num_workers : int
        number of workers to be used for dataloader (default: 4)

    -------
    Returns
    -------
    (train_dataset_loader, valid_dataset_loader) : tuple
        tuple of torch dataloaders
    """
    list_images = sorted(
        [
            f
            for f in os.listdir(os.path.join(dir_dataset, "train", "images"))
            if f.endswith(".jpg")
        ]
    )
    list_train_images, list_valid_images = train_test_split(
        list_images,
        test_size=0.05,
        shuffle=True,
        random_state=random_state,
    )
    print("dataset information")
    print(f"number of train samples: {len(list_train_images)}")
    print(f"number of validation samples: {len(list_valid_images)}")

    train_dataset = M4DSAROilSpillDataset(
        os.path.join(dir_dataset, "train"), list_train_images, which_set="train"
    )
    train_dataset_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers
    )

    valid_dataset = M4DSAROilSpillDataset(
        os.path.join(dir_dataset, "train"), list_valid_images, which_set="valid"
    )
    valid_dataset_loader = DataLoader(
        valid_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )
    return train_dataset_loader, valid_dataset_loader

def get_dataloader_for_inference(dir_dataset, batch_size=1, num_workers=4):
    """
    ---------
    Arguments
    ---------
    dir_dataset : str
        full path to dataset directory
    batch_size : int
        batch size to be used (default: 1)
    num_workers : int
        number of workers to be used for dataloader (default: 4)

    -------
    Returns
    -------
    (inference_dataset_loader, list_inference_images) : tuple
        tuple of torch dataloader and a list of inference images
    """
    list_inference_images = sorted(
        [
            f
            for f in os.listdir(os.path.join(dir_dataset, "test", "images"))
            if f.endswith(".jpg")
        ]
    )

    inference_dataset = M4DSAROilSpillDataset(
        os.path.join(dir_dataset, "test"), list_inference_images, which_set="test"
    )
    inference_dataset_loader = DataLoader(
        inference_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )
    return inference_dataset_loader, list_inference_images


In [ ]:
class CustomResNet(nn.Module):
    def __init__(
        self,
        layers: List[int],
        block=BasicBlock,
        zero_init_residual=False,
        groups=1,
        num_classes=1000,
        width_per_group=64,
        replace_stride_with_dilation=None,
        norm_layer=None,
    ):
        """
        CustomResNet class to build the CustomResNet encoder model

        ----------
        Attributes
        ----------
        layers : list
            list of number of layers in each residual block
        block : object of block type
            type of the residual block (options = [BasicBlock, Bottleneck])
        zero_init_residual : bool
            to indicate whether to use zero weights for BN
        groups : int
            indicates the number of groups (default: 1)
        num_classes : int
            indicates the number of classes (default: 1000)
        width_per_group : int
            indicates the width per group (default: 64)
        replace_stride_with_dilation : list
            a list indicating whether to replace stride with dilation (default: None)
        norm_layer : object
            object of type batch norm (default: None)
        """

        super(CustomResNet, self).__init__()

        self.dict_encoder_features = {}

        if norm_layer is None:
            self._norm_layer = nn.BatchNorm2d

        self.inplanes = 64
        self.dilation = 1

        if replace_stride_with_dilation is None:
            # each element in the tuple indicates if we should replace
            # the 2x2 stride with a dilated convolution instead
            replace_stride_with_dilation = [False, False, False]

        if len(replace_stride_with_dilation) != 3:
            raise ValueError(
                "replace_stride_with_dilation should be None "
                f"or a 3-element tuple, got {replace_stride_with_dilation}"
            )

        self.groups = groups
        self.base_width = width_per_group

        self.conv1 = nn.Conv2d(
            3, self.inplanes, kernel_size=7, stride=2, padding=3, bias=False
        )
        self.bn1 = self._norm_layer(self.inplanes) #batch normalization layer
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(
            block, 128, layers[1], stride=2, dilate=replace_stride_with_dilation[0]
        )
        self.layer3 = self._make_layer(
            block, 256, layers[2], stride=2, dilate=replace_stride_with_dilation[1]
        )
        self.layer4 = self._make_layer(
            block, 512, layers[3], stride=2, dilate=replace_stride_with_dilation[2]
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        # Zero-initialize the last BN in each residual branch,
        # so that the residual branch starts with zeros, and each residual block behaves like an identity.
        # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)  # type: ignore[arg-type]

    def _make_layer(
        self,
        block,
        planes,
        blocks,
        stride=1,
        dilate=False,
    ):
        norm_layer = self._norm_layer
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                norm_layer(planes * block.expansion),
            )

        layers = []
        layers.append(
            block(
                self.inplanes,
                planes,
                stride,
                downsample,
                self.groups,
                self.base_width,
                previous_dilation,
                norm_layer,
            )
        )
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(
                block(
                    self.inplanes,
                    planes,
                    groups=self.groups,
                    base_width=self.base_width,
                    dilation=self.dilation,
                    norm_layer=norm_layer,
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        """
        ---------
        Arguments
        ---------
        x : torch tensor
            a tensor of input features

        -------
        Returns
        -------
        x : torch tensor
            output of the CustomResNet
        """
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        self.dict_encoder_features["block_1"] = x

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        return x


def _resnet(block_type, layers, weights=None, progress=True):
    """
    ---------
    Arguments
    ---------
    block_type : object
        object of type block
    layers : list
        list of layers in each residual block
    weights : object
        object of type ResNet weights
    progress : bool
        indicates whether to show progress or not

    -------
    Returns
    -------
    model : object
        model object of type CustomResNet
    """
    model = CustomResNet(layers, block_type)

    if weights is not None:
        model.load_state_dict(weights.get_state_dict(progress=progress))

    return model

In [ ]:
def resnet34(pretrained=True):
    r"""ResNet-34 model from
    `"Deep Residual Learning for Image Recognition" <https://arxiv.org/pdf/1512.03385.pdf>`_

    ---------
    Arguments
    ---------
    pretrained : bool
        if True, returns a model pre-trained on ImageNet
    """
    if pretrained:
        weights = ResNet34_Weights.IMAGENET1K_V1
    else:
        weights = None
    return _resnet(BasicBlock, [3, 4, 6, 3], weights=weights)

In [ ]:
class DeepLabV3Plus(nn.Module):
    """
    DeepLabV3Plus class to build the DeepLabV3+ decoder model

    ----------
    Attributes
    ----------
    in_channels : int
        number of input channels to decoder model from the encoder model's output
    encoder_channels : int
        number of channels from the intermediate layer of the encoder for merging
    num_classes : int
        number of classes for which the decoder needs to be built
    encoder_projection_channels : int
        number of resulting projection channels from the intermediate layer of the encoder for merging (default: 48)
    aspp_out_channels : int
        number of output channels of the ASPP layer (default: 256)
    final_out_channels : int
        number of output channels before applying classification conv layer (default: 256)
    aspp_dilate: list
        a list of dilation rates to be used for conv layers in ASPP block (default: [12, 24, 36])
    """

    def __init__(
        self,
        in_channels,
        encoder_channels,
        num_classes,
        encoder_projection_channels=48,
        aspp_out_channels=256,
        final_out_channels=256,
        aspp_dilate=[12, 24, 36],
    ):

        super().__init__()

        # 1. Projection convolution: Converts encoder output from an intermediate layer into a smaller number of channels
        self.projection_conv = nn.Sequential(
            nn.Conv2d(encoder_channels, encoder_projection_channels, 1, bias=False),  # 1x1 convolution to reduce channels
            nn.BatchNorm2d(encoder_projection_channels),  # Batch normalization
            nn.ReLU(inplace=True),  # ReLU activation
        )

        # 2. ASPP Block: Atrous Spatial Pyramid Pooling block for multi-scale feature extraction
        self.aspp_block = ASPPBlock(
            in_channels, aspp_dilate, aspp_out_channels=aspp_out_channels
        )

        # 3. Classifier Convolution Block: Combines the ASPP output and the projected encoder features
        self.classifier_conv_block = nn.Sequential(
            nn.Conv2d(
                aspp_out_channels + encoder_projection_channels,  # Concatenated channel depth
                final_out_channels,  # Reducing it back to the final output channels
                3,  # 3x3 convolution
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(final_out_channels),  # Batch normalization
            nn.ReLU(inplace=True),  # ReLU activation
            nn.Conv2d(final_out_channels, num_classes, 1, stride=1, padding="same"),  # Final convolution to get class logits
        )

        # 4. Initialize the weights of the network
        self._init_weights()

    def _init_weights(self):
        """
        Initializes the weights of the network with Kaiming Normal for Conv2d layers and sets BatchNorm and GroupNorm weights to 1 and biases to 0.
        """
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)  # Kaiming Normal initialization for Conv2d
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)  # Initialize BatchNorm/GroupNorm weights to 1
                nn.init.constant_(m.bias, 0)  # Initialize biases to 0
        return

    def forward(self, encoded_features, block_1_features):
        """
        ---------
        Arguments
        ---------
        encoded_features : torch tensor
            A tensor of encoded features from the encoder.
        block_1_features : torch tensor
            A tensor of features from the intermediate layer from the encoder.

        -------
        Returns
        -------
        final_output_feature : torch tensor
            A tensor of final output logits.
        """

        # 1. Project the intermediate encoder features to reduce the number of channels
        encoder_connection = self.projection_conv(block_1_features)

        # 2. Process the final output of the encoder through the ASPP block
        aspp_output_feature = self.aspp_block(encoded_features)

        # 3. Upsample the ASPP output to match the spatial dimensions of the encoder connection features
        aspp_output_feature = F.interpolate(
            aspp_output_feature,
            size=encoder_connection.shape[2:],  # Matching the height and width of encoder connection
            mode="bilinear",
            align_corners=False,
        )

        # 4. Concatenate the ASPP output with the encoder connection features
        final_output_feature = self.classifier_conv_block(
            torch.cat([encoder_connection, aspp_output_feature], dim=1)
        )

        # 5. Return the final output logits
        return final_output_feature

In [ ]:
class ASPPConvLayer(nn.Sequential):
    """
    ASPPConvLayer class to build the ASPPConvLayer used in ASPPBlock

    ----------
    Attributes
    ----------
    in_channels : int
        number of input channels to ASPPConvLayer
    out_channels : int
        number of output channels from ASPPConvLayer
    dilation : int
        dilation rate
    """

    def __init__(self, in_channels, out_channels, dilation):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                3,
                padding=dilation,
                dilation=dilation,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_normal_(m.weight)
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        return

    def forward(self, x):
        """
        ---------
        Arguments
        ---------
        x : torch tensor
            a tensor of input features

        -------
        Returns
        -------
        x : torch tensor
            output of the ASPPConvLayer
        """
        x = self.conv_block(x)
        return x


class ASPPPoolingLayer(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        """
        ASPPPoolingLayer class to build the ASPPPoolingLayer used in ASPPBlock

        ----------
        Attributes
        ----------
        in_channels : int
            number of input channels to ASPPPoolingLayer
        out_channels : int
            number of output channels from ASPPPoolingLayer
        """
        super().__init__()
        self.avg_pool_block = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(
                in_channels, out_channels, 1, stride=1, padding="same", bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_normal_(m.weight)
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        return

    def forward(self, x):
        """
        ---------
        Arguments
        ---------
        x : torch tensor
            a tensor of input features

        -------
        Returns
        -------
        x : torch tensor
            output of the ASPPPoolingLayer
        """
        size = x.shape[2:]
        x = self.avg_pool_block(x)
        x = F.interpolate(x, size=size, mode="bilinear", align_corners=False)
        return x



class ASPPBlock(nn.Module):
    def __init__(self, in_channels, atrous_rates, aspp_out_channels=256):
        """
        ASPPBlock class to build the ASPPBlock

        ---------
        Attributes
        ----------
        in_channels : int
            number of input channels to ASPPBlock
        atrous_rates : list
            list of dilation rates
        aspp_out_channels : int
            number of output channels of the ASPPBlock
        """
        super().__init__()

        self.aspp_init_conv = nn.Sequential(
            nn.Conv2d(
                in_channels, aspp_out_channels, 1, stride=1, padding="same", bias=False
            ),
            nn.BatchNorm2d(aspp_out_channels),
            nn.ReLU(inplace=True),
        )

        modules = []
        modules.append(self.aspp_init_conv)
        modules += [
            ASPPConvLayer(in_channels, aspp_out_channels, atrous_rate)
            for atrous_rate in atrous_rates
        ]
        modules.append(ASPPPoolingLayer(in_channels, aspp_out_channels))
        self.aspp_module_layers = nn.ModuleList(modules)

        self.aspp_final_conv = nn.Sequential(
            nn.Conv2d(
                (2 + len(atrous_rates)) * aspp_out_channels,
                aspp_out_channels,
                1,
                stride=1,
                padding="same",
                bias=False,
            ),
            nn.BatchNorm2d(aspp_out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_normal_(m.weight)
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        return

    def forward(self, x):
        """
        ---------
        Arguments
        ---------
        x : torch tensor
            a tensor of input features

        -------
        Returns
        -------
        x : torch tensor
            output of the ASPPBlock
        """
        aspp_outputs = []
        for aspp_layer in self.aspp_module_layers:
            aspp_outputs.append(aspp_layer(x))
        concat_aspp_output = torch.cat(aspp_outputs, dim=1)
        final_aspp_output = self.aspp_final_conv(concat_aspp_output)
        return final_aspp_output

In [ ]:
class ResNet34DeepLabV3Plus(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        """
        ----------
        Attributes
        ----------
        num_classes : int
            number of classes in the dataset
        pretrained : bool
            indicates whether to load pretrained weights for the encoder model (default: True)
        """
        super().__init__()

        self.encoder = resnet34(pretrained=pretrained)
        self.segmenter = DeepLabV3Plus(
            in_channels=512, encoder_channels=64, num_classes=num_classes
        )

    def forward(self, x):
        input_shape = x.shape[2:]#gets the Height and Width HxW
        encoded_features = self.encoder(x)
        x = self.segmenter(
            encoded_features, self.encoder.dict_encoder_features["block_1"]
        )
        x = F.interpolate(x, size=input_shape, mode="bilinear", align_corners=False)
        return x


In [ ]:
def compute_mean_pixel_acc(true_label, pred_label):
    """
    ---------
    Arguments
    ---------
    true_label : ndarray
        a numpy array of groundtruth label
    pred_label : ndarray
        a numpy array of prediction label

    -------
    Returns
    -------
    mean_pixel_accuracy : float
        mean pixel accuracy
    """
    if true_label.shape != pred_label.shape:
        print(
            "true_label has dimension",
            true_label.shape,
            ", pred_label values have shape",
            pred_label.shape,
        )
        return

    if true_label.dim() != 3:
        print("true_label has dim", true_label.dim(), ", Must be 3.")
        return

    acc_sum = 0
    for i in range(true_label.shape[0]):
        true_label_arr = true_label[i, :, :].clone().detach().cpu().numpy()
        pred_label_arr = pred_label[i, :, :].clone().detach().cpu().numpy()
        true_label_arr = true_label_arr.astype(np.int32)
        pred_label_arr = pred_label_arr.astype(np.int32)

        same = (true_label_arr == pred_label_arr).sum()

        a, b = true_label_arr.shape
        total = a * b

        acc_sum += same / total

    mean_pixel_accuracy = acc_sum / true_label.shape[0]
    return mean_pixel_accuracy


# compute mean IOU
def compute_mean_IOU(true_label, pred_label, num_classes=5):
    """
    ---------
    Arguments
    ---------
    true_label : ndarray
        a numpy array of groundtruth label
    pred_label : ndarray
        a numpy array of prediction label
    num_classes : int
        number of classes in the dataset (default: 5)

    -------
    Returns
    -------
    mean_iou : float
        mean IoU
    """
    iou_list = list()
    present_iou_list = list()

    pred_label = pred_label.view(-1)
    true_label = true_label.view(-1)
    # Note: Following for loop goes from 0 to (num_classes-1)
    # in computation of IoU.
    for sem_class in range(num_classes):
        pred_label_inds = pred_label == sem_class
        target_inds = true_label == sem_class
        if target_inds.long().sum().item() == 0:
            iou_now = float("nan")
        else:
            intersection_now = (pred_label_inds[target_inds]).long().sum().item()
            union_now = (
                pred_label_inds.long().sum().item()
                + target_inds.long().sum().item()
                - intersection_now
            )
            iou_now = float(intersection_now) / float(union_now)
            present_iou_list.append(iou_now)
        iou_list.append(iou_now)
    present_iou_list = np.array(present_iou_list)
    return np.mean(present_iou_list)

def compute_mean_IOU(true_label, pred_label, num_classes=5):
    """
    ---------
    Arguments
    ---------
    true_label : ndarray
        a numpy array of groundtruth label
    pred_label : ndarray
        a numpy array of prediction label
    num_classes : int
        number of classes in the dataset (default: 5)

    -------
    Returns
    -------
    mean_iou : float
        mean IoU
    """
    iou_list = list()
    present_iou_list = list()

    pred_label = pred_label.view(-1)
    true_label = true_label.view(-1)
    # Note: Following for loop goes from 0 to (num_classes-1)
    # in computation of IoU.
    for sem_class in range(num_classes):
        pred_label_inds = pred_label == sem_class
        target_inds = true_label == sem_class
        if target_inds.long().sum().item() == 0:
            iou_now = float("nan")
        else:
            intersection_now = (pred_label_inds[target_inds]).long().sum().item()
            union_now = (
                pred_label_inds.long().sum().item()
                + target_inds.long().sum().item()
                - intersection_now
            )
            iou_now = float(intersection_now) / float(union_now)
            present_iou_list.append(iou_now)
        iou_list.append(iou_now)
    present_iou_list = np.array(present_iou_list)
    return np.mean(present_iou_list)


def compute_class_IOU(true_label, pred_label, num_classes=5):
    """
    ---------
    Arguments
    ---------
    true_label : ndarray
        a numpy array of groundtruth label
    pred_label : ndarray
        a numpy array of prediction label
    num_classes : int
        number of classes in the dataset (default: 5)


    -------
    Returns
    -------
    per_class_iou : ndarray
        a numpy array of per class IoU
    """
    iou_list = list()
    present_iou_list = list()

    pred_label = pred_label.view(-1)
    true_label = true_label.view(-1)

    per_class_iou = np.zeros(num_classes)

    # Note: Following for loop goes from 0 to (num_classes-1)
    # in computation of IoU.
    for sem_class in range(num_classes):
        pred_label_inds = pred_label == sem_class
        target_inds = true_label == sem_class
        if target_inds.long().sum().item() == 0:
            iou_now = float("nan")
        else:
            intersection_now = (pred_label_inds[target_inds]).long().sum().item()
            union_now = (
                pred_label_inds.long().sum().item()
                + target_inds.long().sum().item()
                - intersection_now
            )
            iou_now = float(intersection_now) / float(union_now)
            present_iou_list.append(iou_now)
        per_class_iou[sem_class] = iou_now
    return per_class_iou


In [ ]:
from torch.optim.lr_scheduler import _LRScheduler
def validation_loop(dataset_loader, model, ce_loss, device):
    """
    ---------
    Arguments
    ---------
    dataset_loader : object
        object of type dataloader
    model : object
        object of type model
    ce_loss : object
        object of type cross entropy loss
    device : str
        device on which training needs to be run

    -------
    Returns
    -------
    (valid_loss, valid_acc, valid_IOU) : tuple
        a tuples of torch floats of mean loss, mean accuracy, mean IoU for the validation set
    """
    model.eval()
    size = len(dataset_loader.dataset)
    num_batches = len(dataset_loader)
    valid_loss, valid_acc, valid_IOU = 0, 0, 0

    with torch.no_grad():
        for image, label in dataset_loader:
            image = image.to(device, dtype=torch.float)
            label = label.to(device, dtype=torch.long)

            pred_logits = model(image)
            valid_loss += ce_loss(pred_logits, label)

            pred_probs = F.softmax(pred_logits, dim=1)
            pred_label = torch.argmax(pred_probs, dim=1)

            valid_acc += compute_mean_pixel_acc(label, pred_label)
            valid_IOU += compute_mean_IOU(label, pred_label)

    valid_loss /= num_batches
    valid_acc /= num_batches
    valid_IOU /= num_batches
    return valid_loss, valid_acc, valid_IOU


def train_loop(dataset_loader, model, ce_loss, optimizer, device):
    """
    ---------
    Arguments
    ---------
    dataset_loader : object
        object of type dataloader
    model : object
        object of type model
    ce_loss : object
        object of type cross entropy loss
    optimizer : object
        object of type optimizer
    device : str
        device on which training needs to be run

    -------
    Returns
    -------
    train_loss : torch float
        mean loss for the training set
    """
    model.train()
    size = len(dataset_loader.dataset)
    num_batches = len(dataset_loader)
    train_loss = 0

    for image, label in dataset_loader:
        image = image.to(device, dtype=torch.float)
        label = label.to(device, dtype=torch.long)
        optimizer.zero_grad()

        pred_logits = model(image)
        loss = ce_loss(pred_logits, label)

        # Backpropagation
        loss.backward()
        optimizer.step()

        train_loss += loss
    train_loss /= num_batches
    return train_loss

class PolynomialLR(_LRScheduler):
    """
    PolynomialLR class for the polynomial learning rate scheduler

    ----------
    Attributes
    ----------
    optimizer : object
        object of type optimizer
    max_epochs : int
        maximum number of epochs for which optimization needs to be run
    power : float
        the power term in the polynomial learning rate scheduler (default: 0.9)
    last_epoch : int
        last epoch in the optimization (default: -1)
    min_lr : float
        minimum value for the learning rate (default: 1e-6)
    """

    def __init__(self, optimizer, max_epochs, power=0.9, last_epoch=-1, min_lr=1e-6):
        self.power = power
        self.max_epochs = max_epochs
        self.min_lr = min_lr  # avoid zero lr
        super(PolynomialLR, self).__init__(optimizer, last_epoch)

    def get_lr(self):
        return [
            max(
                base_lr * (1 - self.last_epoch / self.max_epochs) ** self.power,
                self.min_lr,
            )
            for base_lr in self.base_lrs
        ]


In [ ]:
!cp "/content/drive/Othercomputers/MiPC/Code_OilSpill/training/logger_utils.py" "/content/"
from logger_utils import CSVWriter, write_dict_to_json

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.epochs_no_improve = 0
        self.early_stop = False

    def __call__(self, validation_loss):
        if self.best_score is None:
            self.best_score = validation_loss
        elif validation_loss < self.best_score - self.min_delta:
            self.best_score = validation_loss
            self.epochs_no_improve = 0
        else:
            self.epochs_no_improve += 1
            if self.epochs_no_improve >= self.patience:
                self.early_stop = True

In [ ]:
early_stopping = EarlyStopping(patience=5, min_delta=0.001)

def batch_train(FLAGS):
    dir_path = os.path.join(FLAGS.dir_model, FLAGS.which_model)
    if not os.path.isdir(dir_path):
        os.makedirs(dir_path)
        print(f"created directory : {dir_path}")
    csv_writer = CSVWriter(
        file_name=os.path.join("/content/", "train_metrics.csv"),
        column_names=["epoch", "train_loss", "valid_loss", "valid_acc", "valid_IOU"],
    )

    os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dataset_loader, valid_dataset_loader = get_dataloaders_for_training(
        FLAGS.dir_dataset,
        FLAGS.batch_size,
        random_state=FLAGS.random_state,
    )

    # Model selection
    oil_spill_seg_model = ResNet34DeepLabV3Plus(
        num_classes=FLAGS.num_classes, pretrained=bool(FLAGS.pretrained)
    )
    oil_spill_seg_model.to(device)

    # Optimizer selection
    if FLAGS.which_optimizer == "sgd":
        optimizer = torch.optim.SGD(
            oil_spill_seg_model.parameters(),
            lr=FLAGS.learning_rate,
            momentum=0.9,
            weight_decay=FLAGS.weight_decay,
        )
        lr_scheduler = PolynomialLR(
            optimizer,
            FLAGS.num_epochs + 1,
            power=0.9,
        )
    elif FLAGS.which_optimizer == "adamw":
        optimizer = torch.optim.AdamW(
            oil_spill_seg_model.parameters(),
            lr=FLAGS.learning_rate,
            weight_decay=FLAGS.weight_decay,
        )

    ce_loss = torch.nn.CrossEntropyLoss()
    print(f"\ntraining oil spill segmentation model: {FLAGS.which_model}\n")

    # Manually creating a dictionary of attributes that are JSON serializable
    serializable_flags = {k: v for k, v in vars(FLAGS).items() if isinstance(v, (int, float, str, list, dict))}

    write_dict_to_json(os.path.join(dir_path, "params.json"), serializable_flags)

    for epoch in range(1, FLAGS.num_epochs + 1):
        t_1 = time.time()
        train_loss = train_loop(
            train_dataset_loader, oil_spill_seg_model, ce_loss, optimizer, device
        )
        t_2 = time.time()
        print("-" * 100)
        print(
            f"Epoch : {epoch}/{FLAGS.num_epochs}, time: {(t_2-t_1):.2f} sec., train loss: {train_loss:.5f}"
        )
        valid_loss, valid_acc, valid_IOU = validation_loop(
            valid_dataset_loader, oil_spill_seg_model, ce_loss, device
        )
        print(
            f"validation loss: {valid_loss:.5f}, validation accuracy: {valid_acc:.5f}, validation IOU: {valid_IOU:.5f}"
        )
        early_stopping(valid_loss)
        if early_stopping.early_stop:
          print("Early stopping")
          break

        csv_writer.write_row(
            [
                epoch,
                np.around(train_loss.cpu().detach().numpy(), 5),
                np.around(valid_loss.cpu().detach().numpy(), 5),
                round(valid_acc, 5),
                round(valid_IOU, 5),
            ]
        )
        torch.save(
            oil_spill_seg_model.state_dict(),
            os.path.join(dir_path, f"oil_spill_seg_{FLAGS.which_model}_{epoch}.pt"),
        )
        if FLAGS.which_optimizer == "sgd":
            lr_scheduler.step()
    print("Training complete!!!!")
    csv_writer.close()
    return

In [ ]:
import os

class FLAGS:
    # Set all necessary attributes manually
    dir_dataset = "/content/drive/MyDrive/CIMA2023/Documentos2023/Proyectos/Proy1-ImagenesEspectrales/data/imagenes/OilDataset/"
    pretrained = 1  # 1 for True, 0 for False
    random_state = 3
    which_optimizer = "sgd"  # or "adamw"
    learning_rate = 1e-2  # Set your learning rate
    weight_decay = 1e-4  # Set your weight decay
    num_epochs = 50  # Set the number of epochs
    batch_size = 4  # Set the batch size
    num_classes = 5  # Set the number of classes
    which_model = "resnet_34_deeplab_v3+"  # Specify the model
    dir_model = os.getcwd()  # Directory to save models

# Since all parameters are already set, there is no need to reassign them.


In [ ]:
# Call the batch_train function with the FLAGS object
import time

batch_train(FLAGS)


/content/train_metrics.csv created successfully with header row
dataset information
number of train samples: 951
number of validation samples: 51

training oil spill segmentation model: resnet_34_deeplab_v3+

----------------------------------------------------------------------------------------------------
Epoch : 1/50, time: 31.87 sec., train loss: 0.31817
validation loss: 0.19349, validation accuracy: 0.93189, validation IOU: 0.45750
----------------------------------------------------------------------------------------------------
Epoch : 2/50, time: 32.21 sec., train loss: 0.21008
validation loss: 0.20782, validation accuracy: 0.92144, validation IOU: 0.50281
----------------------------------------------------------------------------------------------------
Epoch : 3/50, time: 32.11 sec., train loss: 0.17412
validation loss: 0.15554, validation accuracy: 0.93703, validation IOU: 0.51079
--------------------------------------------------------------------------------------------

In [ ]:
import os
import numpy as np
import torch
from skimage.io import imsave
import torch.nn.functional as F


def create_directory(dir_path):
    """Create a directory if it doesn't exist."""
    if not os.path.isdir(dir_path):
        os.makedirs(dir_path)
        print(f"Created directory: {dir_path}")

def inference_loop(
    dataset_loader, list_images, model, dir_labels, dir_masks, num_classes, device, image_format=".png"
):
    """Run inference on a subset of images and save the predictions."""
    model.eval()
    infer_acc = 0
    infer_class_IOU = np.array([])

    dict_label_to_color_mapping = {
        0: np.array([0, 0, 0]),
        1: np.array([0, 255, 255]),
        2: np.array([255, 0, 0]),
        3: np.array([153, 76, 0]),
        4: np.array([0, 153, 0]),
    }

    for cur_file_index, (image, label) in enumerate(dataset_loader):
        image = image.to(device, dtype=torch.float)
        label = label.to(device, dtype=torch.long)

        pred_logits = model(image)
        pred_probs = F.softmax(pred_logits, dim=1)
        pred_label = torch.argmax(pred_probs, dim=1)

        infer_acc += compute_mean_pixel_acc(label, pred_label)
        infer_class_IOU_cur_sample = compute_class_IOU(label, pred_label)

        if len(infer_class_IOU) == 0:
            infer_class_IOU = infer_class_IOU_cur_sample
        else:
            infer_class_IOU = np.vstack((infer_class_IOU, infer_class_IOU_cur_sample))

        pred_label_arr = pred_label.detach().cpu().numpy().squeeze()
        pred_label_one_hot = np.eye(num_classes)[pred_label_arr]

        pred_mask_arr = np.zeros((pred_label_arr.shape[0], pred_label_arr.shape[1], 3))
        for sem_class in range(num_classes):
            curr_class_label = pred_label_one_hot[:, :, sem_class]
            curr_class_color_mapping = dict_label_to_color_mapping[sem_class]
            pred_mask_arr += curr_class_label.reshape(pred_label_one_hot.shape[0], pred_label_one_hot.shape[1], 1) * curr_class_color_mapping

        pred_label_arr = pred_label_arr.astype(np.uint8)
        pred_mask_arr = pred_mask_arr.astype(np.uint8)

        file_pred_label = os.path.join(
            dir_labels, list_images[cur_file_index].replace(".jpg", image_format)
        )
        file_pred_mask = os.path.join(
            dir_masks, list_images[cur_file_index].replace(".jpg", image_format)
        )

        padded_height, padded_width = pred_label_arr.shape

        # Remove padding and save the label and mask images
        imsave(file_pred_label, pred_label_arr[11 : padded_height - 11, 15 : padded_width - 15])
        imsave(file_pred_mask, pred_mask_arr[11 : padded_height - 11, 15 : padded_width - 15])

    infer_acc /= len(dataset_loader)
    infer_per_class_IOU = np.nanmean(infer_class_IOU, axis=0)
    return infer_acc, infer_per_class_IOU

def run_inference():
    """Run inference on a subset of images."""
    inference_dataset_loader, list_inference_images = get_dataloader_for_inference(dir_dataset)

    # Select 10 random images for inference
    indices = np.random.choice(len(list_inference_images), 10, replace=False)
    inference_dataset_loader = torch.utils.data.Subset(inference_dataset_loader.dataset, indices)
    list_inference_images = [list_inference_images[i] for i in indices]

    print("Selected 10 random images for inference.")
    print(f"Number of test samples: {len(list_inference_images)}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if which_model == "resnet_34_deeplab_v3+":
        oil_spill_seg_model = ResNet34DeepLabV3Plus(num_classes=num_classes, pretrained=bool(pretrained))
    else:
        print("Model not implemented.")
        return

    oil_spill_seg_model.to(device)
    oil_spill_seg_model.load_state_dict(torch.load(file_model_weights))

    create_directory(dir_labels)
    create_directory(dir_masks)

    infer_acc, infer_per_class_IOU = inference_loop(
        torch.utils.data.DataLoader(inference_dataset_loader, batch_size=1),
        list_inference_images,
        oil_spill_seg_model,
        dir_labels,
        dir_masks,
        num_classes,
        device,
    )

    infer_acc *= 100
    infer_per_class_IOU *= 100
    infer_IOU = np.mean(infer_per_class_IOU)

    print("Inference test set metrics")
    print(f"Accuracy: {infer_acc:.3f} %")
    print(f"Mean IOU: {infer_IOU:.3f} %")
    print("Per class IOU:")
    print(infer_per_class_IOU)

# Define the parameters directly in the notebook
dir_dataset = "/content/drive/MyDrive/CIMA2023/Documentos2023/Proyectos/Proy1-ImagenesEspectrales/data/imagenes/OilDataset/"
num_classes = 5
which_model = "resnet_34_deeplab_v3+"
file_model_weights = "/content/resnet_34_deeplab_v3+/oil_spill_seg_resnet_34_deeplab_v3+_12.pt"
dir_save_preds = "./predictions/"
dir_labels = os.path.join(dir_save_preds, "labels")
dir_masks = os.path.join(dir_save_preds, "masks")
pretrained = 1

# Run inference
run_inference()


Selected 10 random images for inference.
Number of test samples: 10


<ipython-input-40-04742a870db6>:68: UserWarning: ./predictions/labels/img_0077.png is a low contrast image
  imsave(file_pred_label, pred_label_arr[11 : padded_height - 11, 15 : padded_width - 15])
<ipython-input-40-04742a870db6>:68: UserWarning: ./predictions/labels/img_0110.png is a low contrast image
  imsave(file_pred_label, pred_label_arr[11 : padded_height - 11, 15 : padded_width - 15])
<ipython-input-40-04742a870db6>:68: UserWarning: ./predictions/labels/img_0021.png is a low contrast image
  imsave(file_pred_label, pred_label_arr[11 : padded_height - 11, 15 : padded_width - 15])
<ipython-input-40-04742a870db6>:68: UserWarning: ./predictions/labels/img_0049.png is a low contrast image
  imsave(file_pred_label, pred_label_arr[11 : padded_height - 11, 15 : padded_width - 15])
<ipython-input-40-04742a870db6>:69: UserWarning: ./predictions/masks/img_0049.png is a low contrast image
  imsave(file_pred_mask, pred_mask_arr[11 : padded_height - 11, 15 : padded_width - 15])
<ipython-inpu

Inference test set metrics
Accuracy: 88.065 %
Mean IOU: 51.787 %
Per class IOU:
[88.14982541 50.37019249 36.32753178  4.19811321 79.88980716]


<ipython-input-40-04742a870db6>:68: UserWarning: ./predictions/labels/img_0053.png is a low contrast image
  imsave(file_pred_label, pred_label_arr[11 : padded_height - 11, 15 : padded_width - 15])


In [ ]:
inference_dataset_loader, list_inference_images = get_dataloader_for_inference(dir_dataset)
print(list_inference_images)  # Check if the image paths are correct

['img_0001.jpg', 'img_0002.jpg', 'img_0003.jpg', 'img_0004.jpg', 'img_0005.jpg', 'img_0006.jpg', 'img_0007.jpg', 'img_0008.jpg', 'img_0009.jpg', 'img_0010.jpg', 'img_0011.jpg', 'img_0012.jpg', 'img_0013.jpg', 'img_0014.jpg', 'img_0015.jpg', 'img_0016.jpg', 'img_0017.jpg', 'img_0018.jpg', 'img_0019.jpg', 'img_0020.jpg', 'img_0021.jpg', 'img_0022.jpg', 'img_0023.jpg', 'img_0024.jpg', 'img_0025.jpg', 'img_0026.jpg', 'img_0027.jpg', 'img_0028.jpg', 'img_0029.jpg', 'img_0030.jpg', 'img_0031.jpg', 'img_0032.jpg', 'img_0033.jpg', 'img_0034.jpg', 'img_0035.jpg', 'img_0036.jpg', 'img_0037.jpg', 'img_0038.jpg', 'img_0039.jpg', 'img_0040.jpg', 'img_0041.jpg', 'img_0042.jpg', 'img_0043.jpg', 'img_0044.jpg', 'img_0045.jpg', 'img_0046.jpg', 'img_0047.jpg', 'img_0048.jpg', 'img_0049.jpg', 'img_0050.jpg', 'img_0051.jpg', 'img_0052.jpg', 'img_0053.jpg', 'img_0054.jpg', 'img_0055.jpg', 'img_0056.jpg', 'img_0057.jpg', 'img_0058.jpg', 'img_0059.jpg', 'img_0060.jpg', 'img_0061.jpg', 'img_0062.jpg', 'img_00